In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import yaml
import pyopencl.array as clarray
from diffractom import Material, SinglePhaseForwardOperator, Grid, FISTAHuber
from diffractom.operators.single_phase_forward_operator import estimate_L_power
from orix import plot
from orix.quaternion import Orientation, symmetry
from orix.vector import Vector3d
from scipy.spatial.transform import Rotation as R

In [ ]:
data_path = 'examples/example_data.h5'
rotation_matrix_path = 'examples/apriori_grid.npy'
config_path = 'examples/config.yaml'

with open(config_path) as f:
    cfg = yaml.safe_load(f)

N_theta = cfg['N_theta']
N_Omega = cfg['N_Omega']
N_eta = cfg['N_eta']
My = cfg['My']
Nx = cfg['Nx']
Ny = cfg['Ny']

with h5py.File(data_path, "r") as f:
    data        = f["data_array"][...].squeeze()
    A_lattice  = f["lattice"][...]
    hkl_list   = f["hkl_list"][...]
    wavelength = float(f["wavelength"][...])



In [ ]:
mat = Material.from_lattice_parameters(
    lattice_matrix=A_lattice,
    lattice_matrix_kind="direct",
    symmetry_group="cubic",
    wavelength_kev=12.398 / wavelength, # Conversion from Å to keV
    hkl_list=hkl_list,
)

mat.reflections

In [ ]:
sigma_deg = 2.0
sigma_rad = sigma_deg/180.0*np.pi
grid_uniform = Grid.from_random_fundamental_zone(50000, "cubic", sigma_rad)
grid_uniform.prune_close_orientations(theta_deg = 3.0, target=15000)
K_uniform = len(grid_uniform.nodes_at_level(0))
print('#basis functions in uniform grid',K_uniform)

sigma_deg = 0.5
sigma_rad = sigma_deg/180.0*np.pi
rot_mats = np.load(rotation_matrix_path)
grid_tailored = Grid.from_rotation_matrices(rot_mats, sigma_rad)
K_tailored = len(grid_tailored.nodes_at_level(0))
print('#basis functions in uniform grid',K_tailored)


In [ ]:
op = SinglePhaseForwardOperator(
    cfg=cfg, material=mat, grid=grid_tailored,
    max_gb=0.5, verbose=True, normalized=True
)
queue = op.queue

out_cpu = np.ascontiguousarray(data.reshape(N_Omega, My, N_eta * N_theta), dtype=np.float32)
out_gpu = clarray.to_device(queue, out_cpu)

x_gpu   = clarray.zeros(queue, (Nx, Ny, K_tailored), dtype=np.float32, order="F")
weights = np.ones_like(data, dtype=np.float32)
weights[:, :, 40:50,   :] = 0
weights[:, :, 130:140, :] = 0
weights_gpu = clarray.to_device(queue, weights.reshape(N_Omega, My, N_eta * N_theta))



In [ ]:
niter = 100
huber_delta = 30

print("Estimating Lipschitz constant …")
norm_sq = estimate_L_power(op, niter=6, seed=0, eps=1e-30, verbose=1)
print(f"  L ≈ {1.1*norm_sq:.4e}")

print(f"\nRunning FISTA ({niter} iters, huber_delta={huber_delta}) …")
solver = FISTAHuber(
    op, prox_kind="nonneg", L=1.1 * norm_sq,
    huber_delta=huber_delta,
)
solver.run(x_gpu, out_gpu, niter=niter, weights=weights_gpu, verbose=1, diagnostics_interval=5)

In [ ]:
def orientation_matrix_to_rgb(orientation_matrices, axis=np.array([0, 0, 1])):
    """Convert (…, 3, 3) orientation matrices to IPF RGB colours via orix."""
    shape   = orientation_matrices.shape
    u       = orientation_matrices.copy().reshape(-1, 3, 3)
    u       = np.transpose(u, (0, 2, 1))
    ori     = Orientation.from_matrix(u, symmetry.Oh)
    ipfkey  = plot.IPFColorKeyTSL(symmetry.Oh, direction=Vector3d(axis))
    rgb     = ipfkey.orientation2color(ori).reshape(*shape[:-2], 3)
    nan_mask = np.isnan(orientation_matrices[..., 0, 0])
    rgb[nan_mask] = np.nan
    return rgb

def _make_rgba(rgb, mask):
    rgba = np.zeros((*rgb.shape[:2], 4), dtype=np.float32)
    rgba[..., :3] = np.nan_to_num(rgb, nan=0.0)
    rgba[..., 3]  = mask.astype(np.float32)
    return rgba

directions = [([1,0,0],"X"), ([0,1,0],"Y"), ([0,0,1],"Z")]
coeffs = x_gpu.get()
best_k    = np.argmax(coeffs, axis=-1)

mask = coeffs.sum(axis=-1)>0.001
grid_mats   = R.concatenate(grid_tailored.rotations_at_level(0)).as_matrix()
rec_argmax = grid_mats[best_k]
rec_argmax[~mask] = np.nan

for d, lbl in directions:
    rgb = orientation_matrix_to_rgb(rec_argmax, axis=np.array(d))
    rgba = _make_rgba(rgb, mask)[::-1, ::-1]
    fig_s, ax_s = plt.subplots(figsize=(4, 4))
    ax_s.imshow(rgba)
    ax_s.axis("off")


In [ ]:
op = SinglePhaseForwardOperator(
    cfg=cfg, material=mat, grid=grid_uniform,
    max_gb=0.5, verbose=True, normalized=True
)
queue = op.queue

out_cpu = np.ascontiguousarray(data.reshape(N_Omega, My, N_eta * N_theta), dtype=np.float32)
out_gpu = clarray.to_device(queue, out_cpu)

x_gpu   = clarray.zeros(queue, (Nx, Ny, K_uniform), dtype=np.float32, order="F")
weights = np.ones_like(data, dtype=np.float32)
weights[:, :, 40:50,   :] = 0
weights[:, :, 130:140, :] = 0
weights_gpu = clarray.to_device(queue, weights.reshape(N_Omega, My, N_eta * N_theta))



niter = 100
huber_delta = 30

print("Estimating Lipschitz constant …")
norm_sq = estimate_L_power(op, niter=6, seed=0, eps=1e-30, verbose=1)
print(f"  L ≈ {1.1*norm_sq:.4e}")

print(f"\nRunning FISTA ({niter} iters, huber_delta={huber_delta}) …")
solver = FISTAHuber(
    op, prox_kind="nonneg", L=1.1 * norm_sq,
    huber_delta=huber_delta,
)
solver.run(x_gpu, out_gpu, niter=niter, weights=weights_gpu, verbose=1, diagnostics_interval=5)



In [ ]:


def orientation_matrix_to_rgb(orientation_matrices, axis=np.array([0, 0, 1])):
    """Convert (…, 3, 3) orientation matrices to IPF RGB colours via orix."""
    shape   = orientation_matrices.shape
    u       = orientation_matrices.copy().reshape(-1, 3, 3)
    u       = np.transpose(u, (0, 2, 1))
    ori     = Orientation.from_matrix(u, symmetry.Oh)
    ipfkey  = plot.IPFColorKeyTSL(symmetry.Oh, direction=Vector3d(axis))
    rgb     = ipfkey.orientation2color(ori).reshape(*shape[:-2], 3)
    nan_mask = np.isnan(orientation_matrices[..., 0, 0])
    rgb[nan_mask] = np.nan
    return rgb

def _make_rgba(rgb, mask):
    rgba = np.zeros((*rgb.shape[:2], 4), dtype=np.float32)
    rgba[..., :3] = np.nan_to_num(rgb, nan=0.0)
    rgba[..., 3]  = mask.astype(np.float32)
    return rgba

directions = [([1,0,0],"X"), ([0,1,0],"Y"), ([0,0,1],"Z")]
coeffs = x_gpu.get()
best_k    = np.argmax(coeffs, axis=-1)

mask = coeffs.sum(axis=-1)>0.002
grid_mats   = R.concatenate(grid_uniform.rotations_at_level(0)).as_matrix()
rec_argmax = grid_mats[best_k]
rec_argmax[~mask] = np.nan

for d, lbl in directions:
    rgb = orientation_matrix_to_rgb(rec_argmax, axis=np.array(d))
    rgba = _make_rgba(rgb, mask)[::-1, ::-1]
    fig_s, ax_s = plt.subplots(figsize=(4, 4))
    ax_s.imshow(rgba)
    ax_s.axis("off")
